# AtomGPT Reference Hallucination Detector — Evaluation Notebook

This notebook reproduces every benchmark in the paper
*"Hallucination Detector: A hybrid LLM and Semantic Scholar tool calling for
detecting hallucination in scientific literature on AtomGPT.org."*

It calls the public AtomGPT reference-checking API and evaluates it on:

1. The **GPTZero NeurIPS 2025** benchmark (100 confirmed hallucinated citations)
2. A **materials-science pilot** (genuine vs. fabricated materials references)
3. The **CiteAudit** benchmark (balanced genuine + fabricated citations)
4. A **run-to-run determinism** check

Repository: https://github.com/harineralla/AtomGPT-Hallucination-Detector


## 1. Setup

Install dependencies, import libraries, and read the API key securely. The key is
**never hardcoded** — set it as the `AGAPI_KEY` environment variable or enter it
when prompted.

In [ ]:
!pip install -q requests pandas matplotlib numpy

In [ ]:
import os
import json
import time
import getpass
from collections import Counter

import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# API key: read from the environment, or prompt. Never commit a real key.
AGAPI_KEY = os.environ.get("AGAPI_KEY") or getpass.getpass("Enter your AGAPI key: ")

API_URL = "https://atomgpt.org/hallucination/check_stream"
HEADERS = {"Authorization": f"Bearer {AGAPI_KEY}"}

In [ ]:
def check_citation(citation, top_k=5, sleep_time=3, timeout=300):
    """Submit one citation to the AtomGPT checker and return its result dict.

    Returns the object whose ``type == "result"`` from the streaming response,
    or ``None`` if the request failed or returned no result.
    """
    payload = f"References\n1. {citation}"
    r = requests.post(API_URL, headers=HEADERS,
                      data={"text": payload, "top_k": top_k, "sleep_time": sleep_time},
                      timeout=timeout)
    if r.status_code != 200:
        return None
    result = None
    for line in r.text.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        if obj.get("type") == "result":
            result = obj
    return result

## 2. Connectivity check

Confirm the endpoint is reachable and run a single reference end-to-end.

In [ ]:
resp = requests.get("https://atomgpt.org/ai_detector", headers=HEADERS)
print(f"AI Detector endpoint status: {resp.status_code}")

In [ ]:
sample = ("John Doe and Jane Smith. Webvoyager: Building an end-to-end web agent "
          "with large multimodal models. arXiv preprint arXiv:2401.00001, 2024.")

demo = check_citation(sample)
print(json.dumps(demo, indent=2))

## 3. NeurIPS 2025 benchmark — run

Load the 100 confirmed hallucinated citations from the public GPTZero sheet and
score each through the tool. Results are written to `atomgpt_neurips_100.csv`.

In [ ]:
SHEET_ID = "14hIuCK7HCnGhdCuxfwp1wIYbbHMn-K3FNvY9NgWGW0Y"
csv_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv"

df_neurips = pd.read_csv(csv_url)
print(f"Loaded {len(df_neurips)} hallucinated citations")
print(f"Columns: {df_neurips.columns.tolist()}")
df_neurips.head()

In [ ]:
results = []
for idx, row in df_neurips.iterrows():
    citation = row["Example of Verified Hallucination"]
    paper = row["Published Paper"]
    print(f"[{idx + 1}/{len(df_neurips)}] {paper[:60]}...")

    obj = check_citation(citation)
    if obj:
        results.append({
            "published_paper": paper,
            "reference": obj.get("reference", ""),
            "query": obj.get("query", ""),
            "best_match": obj.get("best_match", ""),
            "score": obj.get("score"),
            "title_score": obj.get("title_score"),
            "author_score": obj.get("author_score"),
            "journal_score": obj.get("journal_score"),
            "year_match": obj.get("year_match"),
            "status": obj.get("status", ""),
            "matched_authors": ", ".join(obj.get("matched_authors", [])),
            "matched_venue": obj.get("matched_venue", ""),
            "matched_year": obj.get("matched_year", ""),
            "gptzero_comment": row.get("Comment", ""),
        })
    else:
        results.append({
            "published_paper": paper, "reference": citation, "query": "",
            "best_match": "", "score": None, "title_score": None,
            "author_score": None, "journal_score": None, "year_match": None,
            "status": "error", "matched_authors": "", "matched_venue": "",
            "matched_year": "", "gptzero_comment": row.get("Comment", ""),
        })
    time.sleep(4)  # respect the rate limit

df_results = pd.DataFrame(results)
df_results.to_csv("atomgpt_neurips_100.csv", index=False)
print(f"\nDone. Saved {len(df_results)} results to atomgpt_neurips_100.csv")

## 4. NeurIPS — summary statistics

In [ ]:
print("=== AtomGPT detection summary ===")
print(df_results["status"].value_counts())
print(f"\nMean composite score: {df_results['score'].mean():.1f}")

preview_cols = ["published_paper", "status", "score",
                "title_score", "author_score", "journal_score", "year_match"]
df_results[preview_cols].head(10)

In [ ]:
print(df_results.groupby("status")["score"].describe())

## 5. NeurIPS — status distribution

In [ ]:
COLORS = {
    "likely hallucinated": "#FF4D6A",
    "partial": "#FFB830",
    "verified": "#2EC4B6",
    "error": "#999999",
}
STATUS_ORDER = ["likely hallucinated", "partial", "verified", "error"]

plt.rcParams.update({
    "figure.dpi": 100,
    "font.size": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

counts = df_results["status"].value_counts()
counts = counts.reindex([s for s in STATUS_ORDER if s in counts.index])

bars = ax.bar(counts.index, counts.values,
              color=[COLORS[s] for s in counts.index],
              edgecolor="white", linewidth=1.5, width=0.6)
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha="center", va="bottom", fontweight="bold", fontsize=14)

ax.set_ylabel("Number of Citations")
ax.set_title("AtomGPT Hallucination Detector — Status Distribution\n"
             "(100 Known Hallucinated Citations from NeurIPS 2025)")
ax.set_ylim(0, counts.max() + 10)
plt.tight_layout()
plt.savefig("status_distribution.png", bbox_inches="tight")
plt.show()

## 6. NeurIPS — per-field component analysis

Decompose the composite into its title, author, and journal components to see
which field drives detection. Only scored (non-error) rows are analyzed, and each
component is reported over the subset of citations for which it was extracted.

In [ ]:
df = pd.read_csv("atomgpt_neurips_100.csv")
scored = df[df["status"] != "error"].copy()

comp_cols = ["title_score", "author_score", "journal_score"]
for c in comp_cols:
    scored[c] = pd.to_numeric(scored[c], errors="coerce")

print(f"Scored citations: {len(scored)} of {len(df)}")

In [ ]:
print("=== Component score distribution (scored citations) ===")
print(scored[comp_cols].describe().round(1))

print("\n=== Availability (component present) ===")
for c in comp_cols:
    print(f"{c:14s}: {scored[c].notna().sum()}/{len(scored)} present")

print("\n=== year_match ===")
print(scored["year_match"].value_counts(dropna=False))

print("\n=== Mean component score by status ===")
print(scored.groupby("status")[comp_cols].mean().round(1))

In [ ]:
# What drives a 'likely hallucinated' verdict?
flagged = scored[scored["status"] == "likely hallucinated"]
print(f"=== Failure pattern in {len(flagged)} 'likely hallucinated' citations ===")
for c in comp_cols:
    vals = flagged[c].dropna()
    low = (vals < 30).sum()
    print(f"{c:14s}: {low}/{len(vals)} scored <30  (mean {vals.mean():.1f})")

In [ ]:
# Three separate component distribution figures
panels = [
    ("title_score",   "Title match",   "title_match_dist.png"),
    ("author_score",  "Author match",  "author_match_dist.png"),
    ("journal_score", "Journal match", "journal_match_dist.png"),
]
for col, label, fname in panels:
    vals = scored[col].dropna()
    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.hist(vals, bins=20, range=(0, 100), color="#4C72B0", edgecolor="white")
    ax.axvline(vals.mean(), color="#C44E52", ls="--", lw=1.5,
               label=f"mean {vals.mean():.0f}")
    ax.set_title(f"{label}  (n={len(vals)})", fontsize=11)
    ax.set_xlabel("Score (0-100)")
    ax.set_ylabel("Citations")
    ax.set_xlim(0, 100)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved:", fname)

In [ ]:
# Mean component score by status (grouped bars)
grp = scored.groupby("status")[comp_cols].mean()
order = [s for s in ["likely hallucinated", "partial", "verified"] if s in grp.index]
grp = grp.reindex(order)

x = np.arange(len(grp))
w = 0.25
fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#4C72B0", "#DD8452", "#55A868"]
for i, c in enumerate(comp_cols):
    ax.bar(x + i * w, grp[c], w, label=c.replace("_score", ""),
           color=colors[i], edgecolor="white")
ax.set_xticks(x + w)
ax.set_xticklabels(grp.index, fontsize=9)
ax.set_ylabel("Mean score")
ax.set_title("Component scores by final status")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("component_by_status.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. NeurIPS — AtomGPT vs. GPTZero agreement

GPTZero labels all 100 citations as hallucinated (a single ground-truth class).
This shows how AtomGPT distributes them across its three status classes. Counts are
computed from the scored results, not hardcoded.

In [ ]:
scored_all = df[df["status"] != "error"]
labels = ["likely\nhallucinated", "partial", "verified"]
keys = ["likely hallucinated", "partial", "verified"]
vc = scored_all["status"].value_counts()
values = [int(vc.get(k, 0)) for k in keys]
n_scored = sum(values)

fig, ax = plt.subplots(figsize=(7, 2.2))
data = np.array([values])
ax.imshow(data, cmap="Blues", aspect="auto", vmin=0, vmax=max(values))
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_yticks([0])
ax.set_yticklabels(["GPTZero:\nhallucinated"], fontsize=10)
ax.set_xlabel("AtomGPT predicted status", fontsize=10)
for j, v in enumerate(values):
    share = 100 * v / n_scored if n_scored else 0
    color = "white" if v > max(values) * 0.5 else "black"
    ax.text(j, 0, f"{v}\n({share:.0f}%)", ha="center", va="center",
            fontsize=12, fontweight="bold", color=color)
ax.set_title(f"AtomGPT vs GPTZero on {n_scored} scored hallucinated citations",
             fontsize=11, pad=12)
plt.tight_layout()
plt.savefig("agreement_matrix_1x3.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Materials-science pilot

Score genuine and fabricated materials citations. Upload `materials_citations.json`
(a list of objects with `category`, `label`, and `citation` fields).

In [ ]:
from google.colab import files
files.upload()  # select materials_citations.json

In [ ]:
cites = json.load(open("materials_citations.json"))
print(f"Loaded {len(cites)} materials citations")

mat_results = []
for i, item in enumerate(cites, 1):
    print(f"[{i}/{len(cites)}] ({item['category']}) {item['label']}")
    obj = check_citation(item["citation"])
    mat_results.append({
        "category": item["category"], "label": item["label"],
        "citation": item["citation"],
        "status": obj.get("status", "error") if obj else "error",
        "score": obj.get("score") if obj else None,
        "title_score": obj.get("title_score") if obj else None,
        "author_score": obj.get("author_score") if obj else None,
        "journal_score": obj.get("journal_score") if obj else None,
        "year_match": obj.get("year_match") if obj else None,
        "best_match": obj.get("best_match", "") if obj else "",
    })
    time.sleep(4)

df_mat = pd.DataFrame(mat_results)
df_mat.to_csv("materials_test_results.csv", index=False)
print("\nSaved: materials_test_results.csv")
print(pd.crosstab(df_mat["category"], df_mat["status"]))

## 9. CiteAudit benchmark — run

Score a batch of the balanced CiteAudit sample. The API enforces roughly 100 calls
per session, so the 300-citation sample is split into batches
(`citeaudit_1_100.csv`, `citeaudit_101_200.csv`, `citeaudit_201_300.csv`) and each is
run in a fresh session. Set `BATCH_FILE` to the batch you are running, then merge the
outputs afterward.

In [ ]:
from google.colab import files
files.upload()  # select the batch CSV, e.g. citeaudit_101_200.csv

In [ ]:
BATCH_FILE = "citeaudit_101_200.csv"          # <-- set per batch
OUT_FILE   = BATCH_FILE.replace("citeaudit_", "citeaudit_results_")

df_ca = pd.read_csv(BATCH_FILE)
print(f"Loaded {len(df_ca)} citations from {BATCH_FILE}")

ca_results = []
for i, row in df_ca.iterrows():
    print(f"[{i + 1}/{len(df_ca)}] GT={row['GT_label']}")
    obj = check_citation(row["citation"])
    ca_results.append({
        "idx": row["idx"], "source_type": row["source_type"],
        "GT_label": row["GT_label"], "citation": row["citation"],
        "status": obj.get("status", "error") if obj else "error",
        "score": obj.get("score") if obj else None,
        "title_score": obj.get("title_score") if obj else None,
        "author_score": obj.get("author_score") if obj else None,
        "journal_score": obj.get("journal_score") if obj else None,
        "best_match": obj.get("best_match", "") if obj else "",
    })
    time.sleep(4)

pd.DataFrame(ca_results).to_csv(OUT_FILE, index=False)
print(f"\nSaved {OUT_FILE}")

## 10. CiteAudit — combined status distribution

After running all batches, merge them into `citeaudit_all300_combined.csv` (one row
per citation, with a `status` column) and upload it here to plot the distribution.

In [ ]:
from google.colab import files
files.upload()  # select citeaudit_all300_combined.csv

In [ ]:
cnt = Counter()
with open("citeaudit_all300_combined.csv") as f:
    import csv
    for r in csv.DictReader(f):
        st = (r["status"] or "").strip().lower()
        if st in ("not run", ""):
            st = "error"
        cnt[st] += 1

cats = ["likely hallucinated", "partial", "verified", "error"]
vals = [cnt[c] for c in cats]
colors = ["#FF4B6E", "#FFB22C", "#2EC4B6", "#BFBFBF"]
n = sum(vals)

fig, ax = plt.subplots(figsize=(6.2, 5))
bars = ax.bar(cats, vals, color=colors, width=0.7)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 1.5, str(v),
            ha="center", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Citations")
ax.set_title(f"AtomGPT Hallucination Detector — Status Distribution\n"
             f"({n} CiteAudit Citations)")
ax.set_ylim(0, max(vals) * 1.15)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("citeaudit_status_dist.png", dpi=200, bbox_inches="tight")
plt.show()

## 11. Run-to-run determinism check

Run a fixed subset several times and measure how often the final status label
changes. The scoring pipeline is deterministic, so for a fixed extraction model and
retrieval the labels are expected to be identical across runs.

In [ ]:
N_RUNS = 5

subset = pd.read_csv("citeaudit_all300_combined.csv")
subset = subset[subset["status"] != "not run"].head(20).reset_index(drop=True)

runs = {}
for k in range(N_RUNS):
    labels = []
    for c in subset["citation"]:
        obj = check_citation(c)
        labels.append(obj.get("status", "error") if obj else "error")
        time.sleep(4)
    runs[f"run{k + 1}"] = labels
    print(f"run {k + 1} done")

res = subset[["idx", "GT_label"]].copy()
for k, v in runs.items():
    res[k] = v

run_cols = list(runs.keys())
res["n_distinct"] = res[run_cols].nunique(axis=1)
res["majority"] = res[run_cols].mode(axis=1)[0]
res["agreement"] = res[run_cols].apply(
    lambda r: Counter(r).most_common(1)[0][1] / N_RUNS, axis=1)

unanimous = (res["n_distinct"] == 1).mean()
print(f"\nCitations with identical label across all {N_RUNS} runs: {unanimous:.1%}")
print(f"Mean per-citation label agreement: {res['agreement'].mean():.1%}")
res.to_csv("citeaudit_stability.csv", index=False)